<div dir="rtl" style="text-align: right;">
بخش دوم: شرح مسئله
</div>

<div dir="rtl" style="text-align: right;">
گام اول: شناخت داده و کیفیت آن
</div>

### 1.

In [2]:
import pandas as pd
names_path = "census-income.names"
train_path = "census-income.data"
test_path = "census-income.test"

with open(names_path, "r", encoding="latin-1") as file:
    lines = file.readlines()

start = next(
    i for i, line in enumerate(lines)
    if line.strip().startswith("- 50000")
)

all_columns = []
numeric_features = []
categorical_features = []

for line in lines[start + 1:]:
    line = line.strip()

    if not line or line.startswith("|") or ":" not in line:
        continue

    name, definition = line.split(":", 1)
    name = name.strip()
    definition = definition.strip()

    all_columns.append(name)

    if name == "instance weight":
        continue

    if definition.lower().startswith("continuous"):
        numeric_features.append(name)
    else:
        categorical_features.append(name)

all_columns.append("income")

train_df = pd.read_csv(
    train_path,
    header=None,
    names=all_columns,
    skipinitialspace=True,
    na_values=["?"]
)

test_df = pd.read_csv(
    test_path,
    header=None,
    names=all_columns,
    skipinitialspace=True,
    na_values=["?"]
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nNumeric features:")
print(numeric_features)
print("Number of numeric features:", len(numeric_features))

print("\nCategorical features:")
print(categorical_features)
print("Number of categorical features:", len(categorical_features))

print("\nSample weight:")
print("instance weight")

print("\nTarget label:")
print("income")


Train shape: (199523, 42)
Test shape: (99762, 42)

Numeric features:
['age', 'wage per hour', 'capital gains', 'capital losses', 'dividends from stocks', 'num persons worked for employer', 'weeks worked in year']
Number of numeric features: 7

Categorical features:
['class of worker', 'detailed industry recode', 'detailed occupation recode', 'education', 'enroll in edu inst last wk', 'marital stat', 'major industry code', 'major occupation code', 'race', 'hispanic origin', 'sex', 'member of a labor union', 'reason for unemployment', 'full or part time employment stat', 'tax filer stat', 'region of previous residence', 'state of previous residence', 'detailed household and family stat', 'detailed household summary in household', 'migration code-change in msa', 'migration code-change in reg', 'migration code-move within reg', 'live in this house 1 year ago', 'migration prev res in sunbelt', 'family members under 18', 'country of birth father', 'country of birth mother', 'country of birth

### 2.

In [4]:
def missing_report(df):
    report = pd.DataFrame({
        'Missing Count': df.isnull().sum(),
        'Missing Percentage': df.isnull().mean() * 100
    })

    return report.sort_values(
        by='Missing Count',
        ascending=False
    ).round(2)


print('Training Data:')
train_missing = missing_report(train_df)
display(train_missing)

print('\nTop 5 columns with most missing values:')
display(train_missing.head(5))

print('\nTest Data:')
test_missing = missing_report(test_df)
display(test_missing)

print('\nTop 5 columns with most missing values:')
display(test_missing.head(5))


Training Data:


,Missing Count,Missing Percentage
migration prev res in sunbelt,99696,49.97
migration code-change in msa,99696,49.97
migration code-change in reg,99696,49.97
migration code-move within reg,99696,49.97
country of birth father,6713,3.36
country of birth mother,6119,3.07
country of birth self,3393,1.70
hispanic origin,874,0.44
state of previous residence,708,0.35
age,0,0.00



Top 5 columns with most missing values:


,Missing Count,Missing Percentage
migration prev res in sunbelt,99696,49.97
migration code-change in msa,99696,49.97
migration code-change in reg,99696,49.97
migration code-move within reg,99696,49.97
country of birth father,6713,3.36



Test Data:


,Missing Count,Missing Percentage
migration prev res in sunbelt,49946,50.07
migration code-change in msa,49946,50.07
migration code-change in reg,49946,50.07
migration code-move within reg,49946,50.07
country of birth father,3429,3.44
country of birth mother,3072,3.08
country of birth self,1764,1.77
hispanic origin,405,0.41
state of previous residence,330,0.33
age,0,0.00



Top 5 columns with most missing values:


,Missing Count,Missing Percentage
migration prev res in sunbelt,49946,50.07
migration code-change in msa,49946,50.07
migration code-change in reg,49946,50.07
migration code-move within reg,49946,50.07
country of birth father,3429,3.44


### 3.

In [5]:
def class_distribution(df, dataset_name):

    y = df.iloc[:, 41].astype(str).str.strip()
    weights = pd.to_numeric(df.iloc[:, 24], errors="coerce")

    counts = y.value_counts()
    percentages = counts / len(y) * 100
    weighted_counts = weights.groupby(y).sum()
    weighted_percentages = weighted_counts / weights.sum() * 100

    result = pd.DataFrame({
        "Count": counts,
        "Percentage (%)": percentages,
        "Weighted Count": weighted_counts,
        "Weighted Percentage (%)": weighted_percentages
    })

    print(dataset_name)
    display(result.round(2))

class_distribution(train_df, "Training Data")
class_distribution(test_df, "Test Data")

Training Data


,Count,Percentage (%),Weighted Count,Weighted Percentage (%)
income,,,,
- 50000.,187141,93.79,3.250046e+08,93.59
50000+.,12382,6.21,2.224125e+07,6.41


Test Data


,Count,Percentage (%),Weighted Count,Weighted Percentage (%)
income,,,,
- 50000.,93576,93.8,1.625387e+08,93.66
50000+.,6186,6.2,1.100152e+07,6.34


### 4.

In [6]:
def duplicate_report(df, dataset_name):
    
    duplicate_mask = df.duplicated(keep=False)

    duplicate_records = duplicate_mask.sum()

    duplicate_groups = df.loc[duplicate_mask].drop_duplicates().shape[0]

    feature_cols = [
        i for i in range(df.shape[1])
        if i not in [24, 41]
    ]

    features = df.iloc[:, feature_cols]

    target = df.iloc[:, 41].astype(str).str.strip()

    temp = features.copy()
    temp["_target"] = target

    conflict_mask = (
        temp.groupby(
            features.columns.tolist(),
            dropna=False
        )["_target"].transform("nunique") > 1
    )

    conflict_records = conflict_mask.sum()

    conflict_groups = (
        features.loc[conflict_mask].drop_duplicates().shape[0]
    )

    result = pd.DataFrame({
        "Number of Records": [
            duplicate_records,
            conflict_records
        ],
        "Number of Groups": [
            duplicate_groups,
            conflict_groups
        ]
    }, index=[
        "Completely Duplicate Records",
        "Same Features, Different Labels"
    ])

    print(dataset_name)
    display(result)


duplicate_report(train_df, "Training Data")
duplicate_report(test_df, "Test Data")

Training Data


,Number of Records,Number of Groups
Completely Duplicate Records,5995,2766
"Same Features, Different Labels",379,89


Test Data


,Number of Records,Number of Groups
Completely Duplicate Records,1686,803
"Same Features, Different Labels",119,38


### 5.

In [7]:
def not_in_universe_report(df, dataset_name):

    feature_cols = [
        col for col in df.columns
        if col not in ["instance weight", "income"]
    ]

    nui_counts = df[feature_cols].apply(
        lambda col: col.astype("string").str.strip()
        .eq("Not in universe").sum()
    )

    nui_counts = nui_counts[nui_counts > 0].sort_values(ascending=False)

    missing_counts = df[nui_counts.index].isna().sum()

    result = pd.DataFrame({
        "Not in universe Count": nui_counts,
        "Not in universe (%)": nui_counts / len(df) * 100,
        "Missing Count (?)": missing_counts,
        "Missing (%)": missing_counts / len(df) * 100
    })

    print(dataset_name)
    display(result.round(2))


not_in_universe_report(train_df, "Training Data")
not_in_universe_report(test_df, "Test Data")

Training Data


,Not in universe Count,Not in universe (%),Missing Count (?),Missing (%)
fill inc questionnaire for veteran's admin,197539,99.01,0,0.00
reason for unemployment,193453,96.96,0,0.00
enroll in edu inst last wk,186943,93.69,0,0.00
region of previous residence,183750,92.09,0,0.00
state of previous residence,183750,92.09,708,0.35
member of a labor union,180459,90.45,0,0.00
family members under 18,144232,72.29,0,0.00
major occupation code,100684,50.46,0,0.00
class of worker,100245,50.24,0,0.00
migration prev res in sunbelt,84054,42.13,99696,49.97


Test Data


,Not in universe Count,Not in universe (%),Missing Count (?),Missing (%)
fill inc questionnaire for veteran's admin,98735,98.97,0,0.00
reason for unemployment,96632,96.86,0,0.00
enroll in edu inst last wk,93424,93.65,0,0.00
region of previous residence,92068,92.29,0,0.00
state of previous residence,92068,92.29,330,0.33
member of a labor union,90283,90.50,0,0.00
family members under 18,72394,72.57,0,0.00
major occupation code,50283,50.40,0,0.00
class of worker,50079,50.20,0,0.00
migration prev res in sunbelt,42122,42.22,49946,50.07
